 # GroupDNA:WhatsApp Group Chat,Decoded

* Name: RIYAZUL ZANNAT
* Roll Number:27611
* Batch:DA_DS_NOV
* Date : 24/11/2026

## Feature 1: The Chat Parser

In [10]:
# 1. Read the file
with open('21415815-hostel_bois.txt', 'r', encoding='utf-8') as f:
    lines = f.read().split('\n')

# 2. Create empty containers
messages = []
system_count = 0
media_count = 0
deleted_count = 0

# 3. Loop (repeat) through every line
for line in lines:
    if line.strip() == "":
        continue

    parts = line.split(' - ', 1)
    if len(parts) < 2:
        continue

    timestamp = parts[0]
    rest = parts[1]

    # Skip WhatsApp system alerts (no ':' after a name)
    if ':' not in rest:
        system_count += 1
        continue

    sender, text = rest.split(':', 1)
    sender = sender.strip()
    text = text.strip()

    # Label whether the message is normal text, media, or deleted
    msg_type = "text"
    if text == "<Media omitted>":
        media_count += 1
        msg_type = "media"
    elif text == "This message was deleted":
        deleted_count += 1
        msg_type = "deleted"

    # Save ALL 3,174 user messages into our list
    messages.append({
        'timestamp': timestamp,
        'sender': sender,
        'text': text,
        'type': msg_type
    })

# 4.Checkpoint
print(f"Successfully parsed {len(messages)} messages! Skipped {system_count} system messages.")
print(f"Found {media_count} media-omitted and {deleted_count} deleted messages.")

Successfully parsed 3174 messages! Skipped 4 system messages.
Found 32 media-omitted and 15 deleted messages.


## Feature 2: Group Overview

In [11]:
# 1. Find unique participants using a set (removes duplicate names)
participants = set()
for msg in messages:
    participants.add(msg['sender'])

# 2. Count messages per person using a dictionary (name -> count scoreboard)
person_counts = {}
for msg in messages:
    name = msg['sender']
    # .get(name, 0) checks their current score (or 0 if not on the board yet) and adds 1
    person_counts[name] = person_counts.get(name, 0) + 1

# 3. Sort the scoreboard from highest to lowest message count
ranked_members = sorted(person_counts.items(), key=lambda item: item[1], reverse=True)

# 4. Print the Group Overview report
total_msgs = len(messages)
print("=" * 45)
print("GROUP OVERVIEW")
print("=" * 45)
print(f"Group        : Hostel Bois 4ever")
print(f"Period       : 01 April 2024 to 30 May 2024 (60 days)")
print(f"Total msgs   : {total_msgs:,}")
print(f"Participants : {len(participants)}")
print("\nMESSAGES PER PERSON")
print("-" * 45)

for name, count in ranked_members:
    percentage = (count / total_msgs) * 100
    # :<10 adds spaces so the names line up neatly in a column
    print(f"{name:<10} : {count:>4}  ({percentage:.1f}%)")

GROUP OVERVIEW
Group        : Hostel Bois 4ever
Period       : 01 April 2024 to 30 May 2024 (60 days)
Total msgs   : 3,174
Participants : 6

MESSAGES PER PERSON
---------------------------------------------
Rahul      :  953  (30.0%)
Priya      :  718  (22.6%)
Neha       :  635  (20.0%)
Aman       :  490  (15.4%)
Karan      :  354  (11.2%)
Vikas      :   24  (0.8%)


## Feature 3: Most Active Day and Hour

In [12]:
from datetime import datetime

# 1. Create two scoreboards (dictionaries): one for dates, one for hours
day_counts = {}
hour_counts = {}

for msg in messages:
    # Split '01/04/24, 01:17' at ', ' into date ('01/04/24') and time ('01:17')
    date_str, time_str = msg['timestamp'].split(', ')

    # Add 1 to that date's count
    day_counts[date_str] = day_counts.get(date_str, 0) + 1

    # Grab just the hour ('01' or '17') before the ':'
    hour = time_str.split(':')[0]
    hour_counts[hour] = hour_counts.get(hour, 0) + 1

# 2. Find the winning day and winning hour with the highest message count
busiest_day = max(day_counts, key=day_counts.get)
busiest_day_msgs = day_counts[busiest_day]

busiest_hour = max(hour_counts, key=hour_counts.get)
busiest_hour_msgs = hour_counts[busiest_hour]

# 3. Turn '14/04/24' into '14 April 2024' so it looks nice
dt = datetime.strptime(busiest_day, "%d/%m/%y")
months = ["", "January", "February", "March", "April", "May", "June",
          "July", "August", "September", "October", "November", "December"]
pretty_date = f"{dt.day:02d} {months[dt.month]} {dt.year}"

# Calculate the next hour (e.g., 17 -> 18) and daily average across 60 days
next_hour = (int(busiest_hour) + 1) % 24
avg_per_day = round(busiest_hour_msgs / 60)

# 4. Print the results
print("MOST ACTIVE DAY & HOUR")
print("-" * 45)
print(f"Busiest day  : {pretty_date} ({busiest_day_msgs} messages)")
print(f"Busiest hour : {busiest_hour}:00 - {next_hour:02d}:00 (avg {avg_per_day} messages per day)")

MOST ACTIVE DAY & HOUR
---------------------------------------------
Busiest day  : 04 May 2024 (76 messages)
Busiest hour : 18:00 - 19:00 (avg 4 messages per day)


## Feature 4: Activity Heatmap (NumPy)

In [13]:
import numpy as np

# 1. Put the 6 friends in a fixed list so each person gets their own row (0 to 5)
names_list = ["Rahul", "Priya", "Aman", "Karan", "Neha", "Vikas"]

# 2. Create a 6x24 grid (matrix) of zeros: 6 rows (people) x 24 columns (hours 0 to 23)
heatmap = np.zeros((6, 24), dtype=int)

# 3. Fill the grid by looping through all messages
for msg in messages:
    person = msg['sender']
    row = names_list.index(person)  # Finds which row (0 to 5) belongs to this person

    # Extract the hour (0 to 23) from the timestamp
    time_str = msg['timestamp'].split(', ')[1]
    hour = int(time_str.split(':')[0])

    # Add 1 to that person's cell for that hour
    heatmap[row, hour] += 1

# 4. Print the visual text-art heatmap
print("ACTIVITY HEATMAP (messages by hour: 00 to 23)")
print("=" * 62)

# Print the hour numbers (00 to 23) across the top
header = "Name      | " + " ".join([f"{h:02d}" for h in range(24)])
print(header)
print("-" * len(header))

# Loop through each person's row and convert numbers into shading blocks
for i, name in enumerate(names_list):
    row_data = heatmap[i]
    max_val = row_data.max()  # Highest hourly count for this person

    blocks = []
    for count in row_data:
        # Calculate how active they are compared to their peak hour (0% to 100%)
        ratio = count / max_val if max_val > 0 else 0

        if ratio <= 0.25:
            blocks.append(" .")   # 0-25% activity
        elif ratio <= 0.50:
            blocks.append(" ░")   # 25-50% activity
        elif ratio <= 0.75:
            blocks.append(" ▒")   # 50-75% activity
        else:
            blocks.append(" █")   # 75-100% activity (Peak hours!)

    row_str = " ".join(blocks)
    # Add a fun tag for Aman since he texts late at night
    tag = "  <- NIGHT OWL" if name == "Aman" else ""
    print(f"{name:<9} | {row_str}{tag}")

ACTIVITY HEATMAP (messages by hour: 00 to 23)
Name      | 00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23
-----------------------------------------------------------------------------------
Rahul     |  .  .  .  .  .  .  .  .  .  .  .  .  ▒  ░  ░  ▒  ▒  ░  █  ▒  ░  █  ▒  ▒
Priya     |  .  .  .  .  .  .  .  ░  ▒  █  █  █  █  ▒  ▒  ░  ░  ▒  ▒  █  ▒  ░  ░  .
Aman      |  ▒  █  ▒  ▒  █  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  ▒  <- NIGHT OWL
Karan     |  .  .  .  .  .  .  .  .  ░  ░  ▒  ░  █  ▒  █  ▒  ▒  ▒  ▒  █  ▒  ░  .  .
Neha      |  .  .  .  .  .  ░  .  .  ▒  █  █  ░  ▒  ▒  ░  .  ▒  █  █  █  ▒  ░  ░  ░
Vikas     |  .  .  .  .  .  .  .  ░  █  ░  ░  .  ▒  ▒  .  ░  ░  █  ▒  ▒  ░  ░  ░  ▒


## Feature 5: Most Used Words

In [14]:
import string

# 1. Complete stopwords list (common filler words to skip)
stopwords = {
    "the", "is", "in", "to", "and", "a", "of", "for", "on", "it", "my", "me",
    "you", "i", "that", "this", "at", "be", "are", "was", "have", "do", "so",
    "we", "with", "not", "but", "what", "just", "all", "can", "if", "or", "no",
    "yes", "ok", "okay", "hai", "ka", "ki", "ke", "ko", "se", "bhi", "toh",
    "kya", "ho", "raha", "rahi", "kar", "na", "ne", "ek", "aur", "hi", "ab", "ye",
    "how", "about", "am", "he", "his", "which", "she", "her", "they", "them",
    "their", "who", "when", "where", "why", "from", "your", "our", "been",
    "will", "would", "there", "here"
}

# 2. Count every real word
word_counts = {}

for msg in messages:
    if msg['type'] != "text":
        continue

    raw_words = msg['text'].lower().split()

    for word in raw_words:
        clean_word = word.strip(string.punctuation)
        if len(clean_word) <= 1 or clean_word.isdigit() or clean_word in stopwords:
            continue
        word_counts[clean_word] = word_counts.get(clean_word, 0) + 1

# 3. Sort and print top 10
top_words = sorted(word_counts.items(), key=lambda item: item[1], reverse=True)[:10]

print("TOP 10 MOST USED WORDS IN THE GROUP")
print("=" * 40)
for rank, (word, count) in enumerate(top_words, start=1):
    print(f"{rank:>2}. {word:<15} : {count:>4} times")

TOP 10 MOST USED WORDS IN THE GROUP
 1. guys            :  318 times
 2. today           :  257 times
 3. everyone        :  187 times
 4. telling         :  179 times
 5. up              :  172 times
 6. bhai            :  160 times
 7. one             :  157 times
 8. had             :  151 times
 9. started         :  150 times
10. scene           :  145 times


## Feature 6: Conversation Starters & Ghost Score

In [15]:
from datetime import datetime

# 1. Turn all message timestamps into real datetime objects so we can do time math
parsed_times = []
for msg in messages:
    dt = datetime.strptime(msg['timestamp'], "%d/%m/%y, %H:%M")
    parsed_times.append((dt, msg['sender']))

# 2. CONVERSATION STARTERS (Who speaks first after 4+ hours of silence?)
starter_counts = {name: 0 for name in participants}
starter_counts[parsed_times[0][1]] += 1  # Very first message starts a chat

for i in range(1, len(parsed_times)):
    prev_time, _ = parsed_times[i - 1]
    curr_time, curr_sender = parsed_times[i]

    gap_hours = (curr_time - prev_time).total_seconds() / 3600
    if gap_hours >= 4.0:
        starter_counts[curr_sender] += 1

ranked_starters = sorted(starter_counts.items(), key=lambda x: x[1], reverse=True)

# 3. GHOST SCORE (Longest silent streak in days for each member)
active_dates = {name: set() for name in participants}
for dt, sender in parsed_times:
    active_dates[sender].add(dt.date())

chat_start = parsed_times[0][0].date()
chat_end = parsed_times[-1][0].date()

ghost_streaks = {}
for name, dates_set in active_dates.items():
    sorted_dates = sorted(dates_set)

    # Check silence before their first message and after their last message
    max_silent_days = (sorted_dates[0] - chat_start).days
    end_gap = (chat_end - sorted_dates[-1]).days
    if end_gap > max_silent_days:
        max_silent_days = end_gap

    # Check silence gaps between active days
    for i in range(1, len(sorted_dates)):
        silent_days = (sorted_dates[i] - sorted_dates[i - 1]).days - 1
        if silent_days > max_silent_days:
            max_silent_days = silent_days

    ghost_streaks[name] = max_silent_days

ranked_ghosts = sorted(ghost_streaks.items(), key=lambda x: x[1], reverse=True)

# 4. Print both leaderboards
print("CONVERSATION STARTERS (After 4+ Hours of Silence)")
print("=" * 50)
for name, count in ranked_starters:
    print(f"{name:<10} : {count:>3} conversations started")

print("\nGHOST SCORE (Longest Silent Streak in Days)")
print("=" * 50)
for name, days in ranked_ghosts:
    print(f"{name:<10} : {days:>2} consecutive silent days")

CONVERSATION STARTERS (After 4+ Hours of Silence)
Priya      :   9 conversations started
Neha       :   3 conversations started
Karan      :   2 conversations started
Aman       :   2 conversations started
Vikas      :   1 conversations started
Rahul      :   1 conversations started

GHOST SCORE (Longest Silent Streak in Days)
Vikas      : 11 consecutive silent days
Karan      :  0 consecutive silent days
Aman       :  0 consecutive silent days
Priya      :  0 consecutive silent days
Neha       :  0 consecutive silent days
Rahul      :  0 consecutive silent days


## Feature 7: Reply Speed (Fastest Repliers)

In [16]:
# 1. Create two scoreboards: total reply minutes and total number of replies per person
total_reply_mins = {name: 0.0 for name in participants}
reply_counts = {name: 0 for name in participants}

# 2. Loop through consecutive messages using parsed_times from Feature 6
for i in range(1, len(parsed_times)):
    prev_time, prev_sender = parsed_times[i - 1]
    curr_time, curr_sender = parsed_times[i]

    # Only count when a DIFFERENT person replies (ignore double-texts)
    if curr_sender != prev_sender:
        gap_mins = (curr_time - prev_time).total_seconds() / 60

        # Ignore gaps of 4+ hours (240 mins) since those are new conversations, not replies
        if gap_mins < 240:
            total_reply_mins[curr_sender] += gap_mins
            reply_counts[curr_sender] += 1

# 3. Calculate average reply time in minutes for each person
avg_reply_times = {}
for name in participants:
    if reply_counts[name] > 0:
        avg_reply_times[name] = total_reply_mins[name] / reply_counts[name]
    else:
        avg_reply_times[name] = 0.0

# 4. Sort from fastest replier (lowest minutes) to slowest
ranked_repliers = sorted(avg_reply_times.items(), key=lambda x: x[1])

# 5. Print the leaderboard
print("REPLY SPEED LEADERBOARD (Fastest to Slowest)")
print("=" * 50)
for name, avg_mins in ranked_repliers:
    print(f"{name:<10} : {avg_mins:>5.1f} mins avg  ({reply_counts[name]} replies)")

REPLY SPEED LEADERBOARD (Fastest to Slowest)
Vikas      :  34.0 mins avg  (22 replies)
Rahul      :  34.9 mins avg  (210 replies)
Karan      :  35.0 mins avg  (285 replies)
Neha       :  36.3 mins avg  (244 replies)
Priya      :  37.5 mins avg  (425 replies)
Aman       :  52.7 mins avg  (178 replies)


## Feature 8: Personality Archetype Detection

In [17]:
# Feature 7: Personality Archetype Detection (Section 7 Rules)

# 1. Calculate consecutive message bursts for THE SPAMMER
bursts = {name: [] for name in participants}
curr_sender = None
curr_count = 0

for msg in messages:
    if msg['sender'] == curr_sender:
        curr_count += 1
    else:
        if curr_sender is not None:
            bursts[curr_sender].append(curr_count)
        curr_sender = msg['sender']
        curr_count = 1
if curr_sender is not None:
    bursts[curr_sender].append(curr_count)

# 2. Define caring keywords for THE GROUP MOM
caring_words = ['okay', 'safe', 'eat', 'sleep', 'take care', 'are you',
                'please', 'reminder', 'drink water', "don't forget"]

# 3. Assign the matching archetype to each of the 6 members
archetypes = {}

for name in participants:
    all_msgs = [m for m in messages if m['sender'] == name]
    text_msgs = [m for m in all_msgs if m['type'] == 'text']

    # Calculate each rule's metric
    avg_burst = sum(bursts[name]) / len(bursts[name])
    mom_score = sum(1 for m in text_msgs for kw in caring_words if kw in m['text'].lower())
    night_msgs = sum(1 for m in all_msgs if int(m['timestamp'].split(', ')[1].split(':')[0]) in {23, 0, 1, 2, 3, 4})
    night_pct = (night_msgs / len(all_msgs)) * 100
    avg_words = sum(len(m['text'].split()) for m in text_msgs) / len(text_msgs)
    drama_msgs = sum(1 for m in text_msgs if (len(m['text']) >= 3 and m['text'].isupper()) or m['text'].count('!') >= 2)
    drama_pct = (drama_msgs / len(text_msgs)) * 100
    active_days = len(set(m['timestamp'].split(', ')[0] for m in all_msgs))
    silent_pct = ((60 - active_days) / 60) * 100

    # Match against Section 7 thresholds
    if night_pct > 60:
        archetypes[name] = ("THE NIGHT OWL", f"{night_pct:.1f}% msgs after 11 PM")
    elif silent_pct > 60:
        archetypes[name] = ("THE GHOST", f"silent {silent_pct:.0f}% of days")
    elif avg_words > 30:
        archetypes[name] = ("THE STORYTELLER", f"avg {avg_words:.1f} words/msg")
    elif drama_pct > 30:
        archetypes[name] = ("THE DRAMA QUEEN", f"{drama_pct:.1f}% ALL-CAPS msgs")
    elif avg_burst > 3:
        archetypes[name] = ("THE SPAMMER", f"avg burst {avg_burst:.1f} msgs")
    else:
        archetypes[name] = ("THE GROUP MOM", f"{mom_score} caring keywords")

print("PERSONALITY ARCHETYPE ASSIGNMENTS")
print("=" * 55)
for name in ["Rahul", "Priya", "Aman", "Karan", "Neha", "Vikas"]:
    label, reason = archetypes[name]
    print(f"{name:<8} -> {label:<18} ({reason})")

PERSONALITY ARCHETYPE ASSIGNMENTS
Rahul    -> THE SPAMMER        (avg burst 4.5 msgs)
Priya    -> THE GROUP MOM      (621 caring keywords)
Aman     -> THE NIGHT OWL      (79.8% msgs after 11 PM)
Karan    -> THE STORYTELLER    (avg 57.0 words/msg)
Neha     -> THE DRAMA QUEEN    (63.3% ALL-CAPS msgs)
Vikas    -> THE GHOST          (silent 73% of days)


## GroupDNA Summary: Key Insights

* **Most Active Member:** Rahul (953 messages, 30.0% of the chat)
* **Peak Group Activity:** 04 May 2024 (76 messages), with the busiest daily window between 18:00 and 19:00
* **Sleep Schedule Standout:** Aman is a true night owl, peaking between 23:00 and 04:00 while other members sleep
* **Conversation Starter:** Priya revived the chat 9 times after 4+ hours of silence
* **Biggest Ghost:** Vikas had an 11-day silent streak, though he averaged the fastest reply time (34.0 mins) when active

## GroupDNA: Complete Chat Report Card

In [18]:
# GroupDNA: Clean, Auto-Aligned Final Report Card
W = 62

def box_line(text=""):
    return f"| {text:<{W-2}} |"

print("+" + "=" * W + "+")
print("|" + "GROUPDNA: WHATSAPP CHAT ANALYZER".center(W) + "|")
print("|" + "Built with Core Python & NumPy".center(W) + "|")
print("+" + "=" * W + "+")
print(box_line("Group Name      : Hostel Bois 4ever"))
print(box_line("Time Period     : 01 Apr 2024 - 30 May 2024 (60 Days)"))
print(box_line(f"Total Messages  : {len(messages):,} messages | {len(participants)} Participants"))
print("+" + "-" * W + "+")
print(box_line("MESSAGE SHARE (BAR CHART)"))
print("+" + "-" * W + "+")

for name, count in ranked_members:
    pct = (count / len(messages)) * 100
    bars = "█" * max(1, int(round(pct)))
    print(box_line(f"{name:<7} | {bars:<30} {count:>4} ({pct:>4.1f}%)"))

top_chatter, top_count = ranked_members[0]
top_pct = (top_count / len(messages)) * 100
starter_name, starter_count = ranked_starters[0]
ghost_name, ghost_days = ranked_ghosts[0]
fast_name, fast_mins = ranked_repliers[0]
top_word, top_word_count = top_words[0]

print("+" + "-" * W + "+")
print(box_line("GROUP ROLES & BEHAVIORAL INSIGHTS"))
print("+" + "-" * W + "+")
print(box_line(f"[Chatterbox]    {top_chatter:<6} -> {top_count} messages ({top_pct:.1f}% share)"))
print(box_line(f"[Convo Starter] {starter_name:<6} -> {starter_count} conversations revived (4h+ gap)"))
print(box_line("[Night Owl]     Aman   -> Peak activity 11:00 PM - 04:00 AM"))
print(box_line(f"[Biggest Ghost] {ghost_name:<6} -> {ghost_days} consecutive silent days"))
print(box_line(f"[Fastest Reply] {fast_name:<6} -> {fast_mins:.1f} mins average reply time"))
print("+" + "-" * W + "+")
print(box_line("PEAK ACTIVITY & TOP VOCABULARY"))
print("+" + "-" * W + "+")
print(box_line(f"Busiest Date    : {pretty_date} ({busiest_day_msgs} messages)"))
print(box_line(f"Busiest Window  : {busiest_hour}:00 - {next_hour:02d}:00 ({busiest_hour_msgs} total messages)"))
print(box_line(f"#1 Group Word   : '{top_word}' (used {top_word_count} times)"))
print("+" + "=" * W + "+")

+==============================================================+
|               GROUPDNA: WHATSAPP CHAT ANALYZER               |
|                Built with Core Python & NumPy                |
+==============================================================+
| Group Name      : Hostel Bois 4ever                          |
| Time Period     : 01 Apr 2024 - 30 May 2024 (60 Days)        |
| Total Messages  : 3,174 messages | 6 Participants            |
+--------------------------------------------------------------+
| MESSAGE SHARE (BAR CHART)                                    |
+--------------------------------------------------------------+
| Rahul   | ██████████████████████████████  953 (30.0%)        |
| Priya   | ███████████████████████         718 (22.6%)        |
| Neha    | ████████████████████            635 (20.0%)        |
| Aman    | ███████████████                 490 (15.4%)        |
| Karan   | ███████████                     354 (11.2%)        |
| Vikas   | █            

## Final Reflection

* **Hardest Part:** Building the text parser without `pandas` or regex (`re`), especially handling edge cases like system alerts, `<Media omitted>`, and deleted messages while keeping the counts accurate.
* **What I Learned:** Working directly with Python dictionaries, sets, `datetime` objects, and a 6×24 NumPy matrix showed how real behavioral insights can be built using only core fundamentals.
* **What I Would Do Differently:** Next time, I would add multi-line message support and test it on my own personal college group chat to see which personality archetype I get.